# SensApp quickstart

Store a small temperature dataset, fetch it with Prometheus-style label filters, and plot it. Each run uses a new `run` label, so rerunning the notebook keeps earlier data.

From the repository root, start SensApp in one terminal:

```bash
SENSAPP_STORAGE_CONNECTION_STRING=sqlite://sensapp.db cargo run
```

In another terminal, install the notebook dependencies with [uv](https://docs.astral.sh/uv/) and open it:

```bash
uv venv --python 3.14
uv pip install -e ./python/sensapp jupyterlab matplotlib
.venv/bin/python -m ipykernel install --prefix .venv --name sensapp-quickstart --display-name 'SensApp quickstart'
.venv/bin/jupyter lab python/quickstart.ipynb
```

Select the **SensApp quickstart** kernel. If SensApp runs elsewhere, set `SENSAPP_URL` before starting JupyterLab.

In [ ]:
from datetime import UTC, datetime, timedelta
from urllib.request import Request, urlopen
from uuid import uuid4
import math
import os

import matplotlib.pyplot as plt
from sensapp import SensAppClient

url = os.environ.get('SENSAPP_URL', 'http://127.0.0.1:3000').rstrip('/')
run = uuid4().hex[:8]
now = datetime.now(UTC).replace(microsecond=0)
print(f'SensApp: {url} | run: {run}')

## 1. Store samples

Influx line protocol lets us attach a `room` label to each series. SensApp stores the measurement and field as the metric name `quickstart_temperature value`.

In [ ]:
lines = []
for room, offset in [('kitchen', 0.0), ('office', 2.0)]:
    for i in range(12):
        when = now - timedelta(minutes=5 * (11 - i))
        value = 20 + offset + math.sin(i / 2)
        lines.append(
            f'quickstart_temperature,room={room},run={run} value={value:.2f} {int(when.timestamp())}'
        )

headers = {'Content-Type': 'text/plain'}
if token := os.environ.get('SENSAPP_TOKEN'):
    headers['Authorization'] = f'Bearer {token}'
request = Request(
    f'{url}/api/v2/write?org=quickstart&bucket=quickstart&precision=s',
    data='\n'.join(lines).encode(),
    headers=headers,
    method='POST',
)
with urlopen(request, timeout=10) as response:
    print(f'Stored {len(lines)} samples (HTTP {response.status})')

## 2. Fetch with label filters

`__name__` selects the metric. `run` limits results to this execution. Add `room="kitchen"` to select one room. The `[1h]` range includes the samples above.

In [ ]:
metric = 'quickstart_temperature value'
async with SensAppClient.from_env() as client:
    both = await client.query(f'{{__name__="{metric}",run="{run}"}}[1h]')
    kitchen = await client.query_one(
        f'{{__name__="{metric}",run="{run}",room="kitchen"}}[1h]'
    )

print([(series.labels['room'], len(series)) for series in both])
kitchen.frame.head()

## 3. Plot the result

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for series in both:
    axes[0].plot(series.frame['timestamp'], series.frame['value'], label=series.labels['room'])
axes[0].set(title='Temperature by room', ylabel='°C')
axes[0].legend()

axes[1].plot(kitchen.frame['timestamp'], kitchen.frame['value'], color='tab:blue')
axes[1].set(title='Filtered: kitchen', ylabel='°C')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
plt.show()